# Clinical CVD Model Improvement — Round 2

Your current test results are approximately accuracy 0.707, F1 0.733, ROC-AUC 0.796. This notebook tests alternative preprocessing and models without promising 95%+ performance.

**Integrity rules:** test data is held out; threshold selection uses validation only; no SMOTE before splitting; no test-set tuning. High accuracy cannot be guaranteed, and changing thresholds does not improve ROC-AUC.

In [1]:
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd, joblib
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, balanced_accuracy_score, confusion_matrix, classification_report, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

SEED=42
warnings.filterwarnings("ignore", category=FutureWarning)
CURRENT_DIR=Path.cwd()
PROJECT_ROOT=CURRENT_DIR.parent if CURRENT_DIR.name=="Trained Models" else CURRENT_DIR
# If needed, set the exact project root:
# PROJECT_ROOT=Path(r"C:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14")
XLSX=PROJECT_ROOT/"DataSets"/"Clinical Dataset.xlsx"
CSV=PROJECT_ROOT/"DataSets"/"clinical_dataset_clean.csv"
OUT_DIR=PROJECT_ROOT/"Trained Models"/"clinical_improved_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if XLSX.exists():
    df=pd.read_excel(XLSX); DATA_PATH=XLSX
elif CSV.exists():
    df=pd.read_csv(CSV); DATA_PATH=CSV
else:
    raise FileNotFoundError(f"Dataset not found. Checked: {XLSX} and {CSV}")
print("Project:",PROJECT_ROOT,"\nDataset:",DATA_PATH,"\nShape:",df.shape)
print(df.columns.tolist())

Project: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14 
Dataset: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\DataSets\Clinical Dataset.xlsx 
Shape: (70000, 13)
['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']


## 1. Audit, clean and engineer features

In [2]:
required={"gender","height","weight","ap_hi","ap_lo","cholesterol","gluc","smoke","alco","active","cardio"}
missing=required-set(df.columns)
if missing: raise ValueError(f"Missing columns: {sorted(missing)}")
print("Target distribution before cleaning:"); display(df["cardio"].value_counts(dropna=False).sort_index())
print("Duplicate rows:",df.duplicated().sum())
df=df.drop_duplicates().copy()
if "age_years" not in df:
    if "age" not in df: raise ValueError("Need age (days) or age_years column")
    df["age_years"]=df["age"]/365.25
valid=(df.ap_hi.gt(0)&df.ap_lo.gt(0)&df.ap_hi.le(300)&df.ap_lo.le(200)&df.ap_hi.gt(df.ap_lo)&df.height.between(100,220)&df.weight.between(30,200))
before=len(df); df=df.loc[valid].copy()
print("Rows removed by clinical plausibility filters:",before-len(df),"remaining:",len(df))
df["BMI"]=df.weight/((df.height/100)**2)
df["pulse_pressure"]=df.ap_hi-df.ap_lo
df["MAP"]=(df.ap_hi+2*df.ap_lo)/3
df=df.drop(columns=["id","age","patient_id","record_id","subject_id"],errors="ignore")
if set(df.gender.dropna().unique()).issubset({1,2}): df["gender"]=df.gender.map({1:0,2:1})
df["cardio"]=pd.to_numeric(df.cardio,errors="coerce")
df=df.dropna(subset=["cardio"]).copy(); df["cardio"]=df.cardio.astype(int)
if not set(df.cardio.unique()).issubset({0,1}): raise ValueError("cardio must be binary 0/1")
cats=[c for c in ["cholesterol","gluc"] if c in df]
df=pd.get_dummies(df,columns=cats,drop_first=False,dtype=int)
X=df.drop(columns="cardio").replace([np.inf,-np.inf],np.nan); y=df.cardio
print("Final features:",X.shape,"Target counts:",y.value_counts().to_dict())
display((y.value_counts(normalize=True)*100).round(2))

Target distribution before cleaning:


cardio
0    35021
1    34979
Name: count, dtype: int64

Duplicate rows: 0
Rows removed by clinical plausibility filters: 1325 remaining: 68675
Final features: (68675, 18) Target counts: {0: 34699, 1: 33976}


cardio
0    50.53
1    49.47
Name: proportion, dtype: float64

## 2. Train / validation / test split

In [3]:
X_trainval,X_test,y_trainval,y_test=train_test_split(X,y,test_size=.15,stratify=y,random_state=SEED)
X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=(.15/.85),stratify=y_trainval,random_state=SEED)
features=X_train.columns.tolist()
X_val=X_val.reindex(columns=features); X_test=X_test.reindex(columns=features)
print("Train:",X_train.shape,"Validation:",X_val.shape,"Test:",X_test.shape)

Train: (48071, 18) Validation: (10302, 18) Test: (10302, 18)


## 3. Compare baseline models

In [4]:
prep_scaled=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
prep_tree=Pipeline([("imputer",SimpleImputer(strategy="median"))])
models={
"LogisticRegression":Pipeline([("prep",prep_scaled),("model",LogisticRegression(max_iter=3000,class_weight="balanced",random_state=SEED))]),
"RandomForest":Pipeline([("prep",prep_tree),("model",RandomForestClassifier(n_estimators=400,min_samples_leaf=3,max_features="sqrt",class_weight="balanced_subsample",n_jobs=-1,random_state=SEED))]),
"ExtraTrees":Pipeline([("prep",prep_tree),("model",ExtraTreesClassifier(n_estimators=400,min_samples_leaf=3,max_features="sqrt",class_weight="balanced",n_jobs=-1,random_state=SEED))]),
"HistGradientBoosting":Pipeline([("prep",prep_tree),("model",HistGradientBoostingClassifier(learning_rate=.06,max_iter=300,max_leaf_nodes=15,l2_regularization=1,random_state=SEED))])
}
try:
 from xgboost import XGBClassifier
 models["XGBoost"]=Pipeline([("prep",prep_tree),("model",XGBClassifier(n_estimators=400,max_depth=4,learning_rate=.04,subsample=.85,colsample_bytree=.85,reg_lambda=3,eval_metric="logloss",tree_method="hist",n_jobs=-1,random_state=SEED))])
except ImportError: print("xgboost unavailable; skipped")
try:
 from lightgbm import LGBMClassifier
 models["LightGBM"]=Pipeline([("prep",prep_tree),("model",LGBMClassifier(n_estimators=400,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=30,reg_lambda=3,verbosity=-1,n_jobs=-1,random_state=SEED))])
except ImportError: print("lightgbm unavailable; skipped")
try:
 from catboost import CatBoostClassifier
 models["CatBoost"]=Pipeline([("prep",prep_tree),("model",CatBoostClassifier(iterations=500,depth=6,learning_rate=.04,l2_leaf_reg=5,verbose=False,allow_writing_files=False,random_seed=SEED))])
except ImportError: print("catboost unavailable; skipped")

def probs(model, xx): return model.predict_proba(xx)[:,1]
def score(yy, pp, th=.5):
 pred=(np.asarray(pp)>=th).astype(int)
 return {"threshold":float(th),"accuracy":accuracy_score(yy,pred),"precision":precision_score(yy,pred,zero_division=0),"recall":recall_score(yy,pred,zero_division=0),"f1":f1_score(yy,pred,zero_division=0),"balanced_accuracy":balanced_accuracy_score(yy,pred),"roc_auc":roc_auc_score(yy,pp),"pr_auc":average_precision_score(yy,pp)}
fitted={}; rows=[]
for name,m in models.items():
 print("Fit:",name); m.fit(X_train,y_train); fitted[name]=m
 r={"model":name}; r.update(score(y_val,probs(m,X_val))); rows.append(r)
baseline=pd.DataFrame(rows).sort_values("roc_auc",ascending=False)
display(baseline.round(4))

Fit: LogisticRegression
Fit: RandomForest
Fit: ExtraTrees
Fit: HistGradientBoosting
Fit: XGBoost
Fit: LightGBM
Fit: CatBoost


,model,threshold,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc
4,XGBoost,0.5,0.7336,0.7559,0.6818,0.7169,0.7331,0.7989,0.7817
6,CatBoost,0.5,0.7350,0.7567,0.6845,0.7188,0.7345,0.7985,0.7798
5,LightGBM,0.5,0.7325,0.7543,0.6812,0.7159,0.7319,0.7983,0.7816
3,HistGradientBoosting,0.5,0.7316,0.7535,0.6800,0.7149,0.7311,0.7980,0.7811
2,ExtraTrees,0.5,0.7303,0.7551,0.6733,0.7119,0.7298,0.7957,0.7786
1,RandomForest,0.5,0.7299,0.7455,0.6892,0.7163,0.7294,0.7909,0.7694
0,LogisticRegression,0.5,0.7251,0.7496,0.6673,0.7060,0.7245,0.7885,0.7659


## 4. Tune boosting models by training-only cross-validation

In [5]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
spaces={}
if "LightGBM" in fitted:
 spaces["LightGBM"]={"model__n_estimators":[250,400,600,800],"model__learning_rate":[.015,.025,.04,.06],"model__num_leaves":[7,15,23,31],"model__max_depth":[3,4,5,6,-1],"model__min_child_samples":[15,25,40,60],"model__reg_lambda":[1,3,5,10],"model__class_weight":[None,"balanced"]}
if "XGBoost" in fitted:
 spaces["XGBoost"]={"model__n_estimators":[250,400,600,800],"model__max_depth":[2,3,4,5],"model__learning_rate":[.015,.025,.04,.06],"model__subsample":[.7,.85,1.0],"model__colsample_bytree":[.7,.85,1.0],"model__min_child_weight":[1,3,5,8],"model__reg_lambda":[1,3,5,10]}
if "CatBoost" in fitted:
 spaces["CatBoost"]={"model__iterations":[300,500,700,900],"model__depth":[4,5,6,7,8],"model__learning_rate":[.015,.025,.04,.06],"model__l2_leaf_reg":[3,5,7,10]}
tuned=dict(fitted); tuning=[]
for name,space in spaces.items():
 print("Tune:",name)
 search=RandomizedSearchCV(models[name],space,n_iter=20,scoring="roc_auc",cv=cv,n_jobs=-1,random_state=SEED,refit=True,verbose=1)
 search.fit(X_train,y_train); tuned[name]=search.best_estimator_
 tuning.append({"model":name,"best_cv_roc_auc":search.best_score_,"best_params":search.best_params_})
display(pd.DataFrame(tuning).sort_values("best_cv_roc_auc",ascending=False))

Tune: LightGBM
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Tune: XGBoost
Fitting 5 folds for each of 20 candidates, totalling 100 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

## 5. Select threshold on validation data; do not use test data

In [ ]:
val_rows=[]; thresholds={}
for name,m in tuned.items():
 pp=probs(m,X_val); pr,re,th=precision_recall_curve(y_val,pp)
 f1s=2*pr[:-1]*re[:-1]/(pr[:-1]+re[:-1]+1e-12)
 threshold=float(th[int(np.nanargmax(f1s))]) if len(th) else .5
 thresholds[name]=threshold; row={"model":name}; row.update(score(y_val,pp,threshold)); val_rows.append(row)
val_results=pd.DataFrame(val_rows).sort_values(["roc_auc","f1"],ascending=False)
display(val_results.round(4))
selected_name=str(val_results.iloc[0]["model"]); selected=tuned[selected_name]; selected_threshold=thresholds[selected_name]
print("Selected on validation:",selected_name,"threshold:",selected_threshold)

## 6. Final held-out test evaluation

In [ ]:
test_rows=[]
for name,m in tuned.items():
 row={"model":name}; row.update(score(y_test,probs(m,X_test),thresholds[name])); test_rows.append(row)
test_results=pd.DataFrame(test_rows).sort_values("roc_auc",ascending=False)
display(test_results.round(4))
p_test=probs(selected,X_test); pred=(p_test>=selected_threshold).astype(int)
print("Selected model:",selected_name)
print(classification_report(y_test,pred,target_names=["No CVD","CVD"],zero_division=0))
print("Confusion matrix:\n",confusion_matrix(y_test,pred))
display(pd.Series(score(y_test,p_test,selected_threshold)).round(4))

## 7. Save artifacts

In [ ]:
model_path=OUT_DIR/"clinical_best_model_pipeline.joblib"
joblib.dump(selected,model_path)
joblib.dump(features,OUT_DIR/"clinical_features_v2.pkl")
metadata={"selected_model":selected_name,"threshold":float(selected_threshold),"features":features,"target":"cardio","test_metrics":{k:float(v) for k,v in score(y_test,p_test,selected_threshold).items()},"note":"No metric target is guaranteed; validate on an independent cohort."}
with open(OUT_DIR/"clinical_model_v2_metadata.json","w",encoding="utf-8") as f: json.dump(metadata,f,indent=2)
print("Saved:",model_path)
print("Saved:",OUT_DIR/"clinical_model_v2_metadata.json")

## Important interpretation

- Threshold tuning can change precision, recall, F1 and accuracy, but **cannot improve ROC-AUC**.
- If metrics remain near 0.70–0.80, investigate label definitions, measurement noise, cohort/split differences and external validation.
- Do not use SMOTE before splitting, duplicate records across splits, remove difficult test rows, or keep tuning against test metrics.
- A 95%+ target is not guaranteed. Report the actual results honestly.